PyTorch version: 2.15.0+cpu
CUDA available: False
Training device: CPU

Raw thermal data:


NameError: name 'X' is not defined

In [2]:
import numpy as np
import pandas as pd

data = np.load("../dataset/fir_frames.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
sessions = data["session"]
file_ids = data["file_id"]
classes = data["classes"]

print("Dataset loaded successfully!")
print("Classes:", classes)

Dataset loaded successfully!
Classes: ['walk' 'stand' 'standup' 'sit' 'sitdown' 'falling' 'falled' 'lying'
 'sleeping' 'active']


In [6]:
import os
import pandas as pd
import numpy as np
import torch


windows_metadata = pd.read_csv("../dataset/windows_metadata.csv")

print("Metadata shape:", windows_metadata.shape)
print("Columns:", windows_metadata.columns.tolist())

display(windows_metadata.head())

Metadata shape: (48310, 5)
Columns: ['start', 'binary_label', 'original_label', 'session', 'file_id']


,start,binary_label,original_label,session,file_id
0,32,0,0,20200626_01,0
1,36,0,0,20200626_01,0
2,40,0,0,20200626_01,0
3,44,0,0,20200626_01,0
4,56,0,4,20200626_01,0


In [8]:
WINDOW_SIZE = 16
STRIDE = 4
CENTER_OFFSET = 8
BOUNDARY_MARGIN = 4
FPS_REFERENCE = 7.8

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Center offset:", CENTER_OFFSET)
print("FPS:", FPS_REFERENCE)

Window size: 16
Stride: 4
Center offset: 8
FPS: 7.8


In [9]:
print("X shape:", X.shape)
print("X dtype:", X.dtype)

print("\ny shape:", y.shape)
print("y dtype:", y.dtype)

print("\nWindow size:", WINDOW_SIZE)
print("Stride:", STRIDE)

print("\nBinary label distribution:")
print(windows_metadata["binary_label"].value_counts().sort_index())

print("\nSessions distribution:")
print(windows_metadata.groupby(
    ["session", "binary_label"]
).size().unstack(fill_value=0))

print("\nMissing values:")
print(windows_metadata.isnull().sum())

print("\nStart index range:")
print(windows_metadata["start"].min())
print(windows_metadata["start"].max())

X shape: (228665, 24, 32)
X dtype: float16

y shape: (228665,)
y dtype: int8

Window size: 16
Stride: 4

Binary label distribution:
binary_label
0    42235
1     6075
Name: count, dtype: int64

Sessions distribution:
binary_label      0     1
session                  
20200626_01    4118   605
20200627_01    2748   414
20200628_01    8459  1405
20200628_02    4520   256
20200629_02   17669  3000
20200630_02    4721   395

Missing values:
start             0
binary_label      0
original_label    0
session           0
file_id           0
dtype: int64

Start index range:
32
228631


In [10]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class ThermalFallDataset(Dataset):

    def __init__(self, X, metadata):
        self.X = X
        self.metadata = metadata.reset_index(drop=True)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        row = self.metadata.iloc[idx]

        start = int(row["start"])
        end = start + WINDOW_SIZE

        thermal_window = self.X[start:end].astype(np.float32)

        # Clip temperature values
        thermal_window = np.clip(
            thermal_window, -3, 10
        )

        # Scale values to [0, 1]
        thermal_window = (thermal_window + 3) / 13

        # Shape: [1, 16, 24, 32]
        thermal_window = np.expand_dims(
            thermal_window, axis=0
        )

        label = int(row["binary_label"])

        return (
            torch.tensor(thermal_window, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32)
        )


dataset = ThermalFallDataset(X, windows_metadata)

print("Dataset size:", len(dataset))

sample_X, sample_y = dataset[0]

print("Sample shape:", sample_X.shape)
print("Sample label:", sample_y.item())
print("Minimum:", sample_X.min().item())
print("Maximum:", sample_X.max().item())

Dataset size: 48310
Sample shape: torch.Size([1, 16, 24, 32])
Sample label: 0.0
Minimum: 0.11080228537321091
Maximum: 0.5913461446762085


In [11]:
import torch
import torch.nn as nn

class FallDetection3DCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv3d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d((1, 1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = FallDetection3DCNN()

print(model)

FallDetection3DCNN(
  (features): Sequential(
    (0): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (5): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2), padding=0, dilation=1, ceil_mode=False)
    (8): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (9): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveAvgPool3d(output_size=(1, 1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inpla

In [12]:
sample_input = torch.randn(2, 1, 16, 24, 32)

output = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)

total_params = sum(
    p.numel() for p in model.parameters()
)

print("Total parameters:", total_params)

Input shape: torch.Size([2, 1, 16, 24, 32])
Output shape: torch.Size([2, 1])
Total parameters: 69953


In [13]:
from torch.utils.data import Subset, DataLoader
import numpy as np

TEST_SESSION = "20200626_01"

train_indices = windows_metadata.index[
    windows_metadata["session"] != TEST_SESSION
].tolist()

test_indices = windows_metadata.index[
    windows_metadata["session"] == TEST_SESSION
].tolist()

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)

print("Test session:", TEST_SESSION)
print("Training windows:", len(train_dataset))
print("Testing windows:", len(test_dataset))

print("\nTraining sessions:")
print(
    windows_metadata.iloc[train_indices]["session"]
    .value_counts()
)

print("\nTest labels:")
print(
    windows_metadata.iloc[test_indices]["binary_label"]
    .value_counts()
)

Test session: 20200626_01
Training windows: 43587
Testing windows: 4723

Training sessions:
session
20200629_02    20669
20200628_01     9864
20200630_02     5116
20200628_02     4776
20200627_01     3162
Name: count, dtype: int64

Test labels:
binary_label
0    4118
1     605
Name: count, dtype: int64


In [14]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 1363
Testing batches: 148


In [15]:
train_labels = windows_metadata.iloc[
    train_indices
]["binary_label"].values

n_adl = np.sum(train_labels == 0)
n_fall = np.sum(train_labels == 1)

pos_weight = n_adl / n_fall

print("ADL:", n_adl)
print("FALL:", n_fall)
print("Positive weight:", round(pos_weight, 3))

ADL: 38117
FALL: 5470
Positive weight: 6.968


In [16]:
TEST_SESSION = "20200626_01"
VAL_SESSION = "20200627_01"

train_indices = windows_metadata.index[
    (windows_metadata["session"] != TEST_SESSION) &
    (windows_metadata["session"] != VAL_SESSION)
].tolist()

val_indices = windows_metadata.index[
    windows_metadata["session"] == VAL_SESSION
].tolist()

test_indices = windows_metadata.index[
    windows_metadata["session"] == TEST_SESSION
].tolist()

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)
test_dataset = Subset(dataset, test_indices)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

print("\nTraining sessions:")
print(windows_metadata.iloc[train_indices]["session"].value_counts())

Training: 40425
Validation: 3162
Testing: 4723

Training sessions:
session
20200629_02    20669
20200628_01     9864
20200630_02     5116
20200628_02     4776
Name: count, dtype: int64


In [17]:
import copy
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = FallDetection3DCNN().to(device)

train_labels = windows_metadata.iloc[
    train_indices
]["binary_label"].values

n_adl = np.sum(train_labels == 0)
n_fall = np.sum(train_labels == 1)

pos_weight = torch.tensor(
    [n_adl / n_fall],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

print("Device:", device)
print("Positive weight:", pos_weight.item())

Device: cpu
Positive weight: 6.995450973510742


In [18]:
EPOCHS = 30
PATIENCE = 5

best_val_loss = float("inf")
best_model_state = None
patience_counter = 0

history = []

for epoch in range(EPOCHS):

    # TRAINING
    model.train()

    train_loss = 0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # VALIDATION
    model.eval()

    val_loss = 0
    val_targets = []
    val_scores = []

    with torch.no_grad():

        for batch_X, batch_y in val_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)

            outputs = model(batch_X)

            loss = criterion(outputs, batch_y)

            val_loss += loss.item()

            probabilities = torch.sigmoid(outputs)

            val_scores.extend(
                probabilities.cpu().numpy().flatten()
            )

            val_targets.extend(
                batch_y.cpu().numpy().flatten()
            )

    val_loss /= len(val_loader)

    val_auc = roc_auc_score(
        val_targets, val_scores
    )

    val_ap = average_precision_score(
        val_targets, val_scores
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "val_ap": val_ap
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val AUC: {val_auc:.4f} | "
        f"Val AP: {val_ap:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print("Early stopping triggered.")
        break

model.load_state_dict(best_model_state)

print("\nBest validation loss:", best_val_loss)
print("Training completed.")

KeyboardInterrupt: 